## Setup Tools

In [ ]:
import os
from tavily import TavilyClient
from langchain.tools import tool
from langchain_groq import ChatGroq


tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def web_search(query: str) -> dict:
    """
    Search the web for reliable information.
    """

    try:
        return tavily_client.search(
            query=query,
            search_depth="advanced",
            max_results=5,
        )

    except Exception as e:
        return {"error": str(e)}


## Create State

In [7]:
from langchain.agents import AgentState

class ResearchState(AgentState):
    topic: str
    research_data: str
    image_data: str
    final_answer: str

## Create Subagents

In [12]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile")

ModuleNotFoundError: No module named 'langchain_groq'

In [ ]:
research_agent = create_agent(
    model="llama-3.3-70b-versatile",
    tools=[web_search],
    system_prompt="""
You are a professional research agent your goal is to gather data about palastine history and to be aware of the current situation in palastine.

Your responsibilities:
- Gather accurate information
- Verify claims
- Use reliable sources
- Collect citations
- Summarize findings clearly
- gather information about the current situation in palastine
- you should be very cearful in this topic and make sure to gather information from reliable sources

Rules:
- Always use web search before answering
- Prefer reliable and authoritative sources
- Avoid misinformation
- Make sure you get updated information
- Return:
    1. Key findings
    2. Important facts
    3. Current situation
    4. Sources
"""
)

In [ ]:
image_agent = create_agent(
    model="llama-3.3-70b-versatile",
    system_prompt="""
You are an image research agent.

Your job:
- Suggest visuals related to the topic
- Create detailed image prompts
- Suggest documentary-style visuals
- Avoid graphic or harmful imagery

Return:
- Image title
- Why this image matters
- Search query
- AI image generation prompt
"""
)

In [ ]:
writer_agent = create_agent(
    model="llama-3.3-70b-versatile",
    system_prompt="""
You are a professional writer agent.

Your task:
- Organize research into a clear article
- Write professionally
- Keep structure logical
- Use:
    - Introduction
    - Main sections
    - Conclusion
    - Sources

The writing should:
- Be concise
- Be factual
- Be readable
- Avoid repetition
"""
)

In [ ]:
research_tool = research_agent.as_tool(
    name="research_agent",
    description="Performs deep research on a topic"
)

image_tool = image_agent.as_tool(
    name="image_agent",
    description="Finds or generates relevant images"
)

writer_tool = writer_agent.as_tool(
    name="writer_agent",
    description="Organizes and writes structured content"
)

## ORCHESTRATOR

In [ ]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

In [ ]:
async def manual_orchestrator(user_query: str):

    # STEP 1 — Research
    research_result = await research_agent.ainvoke({
        "messages": [
            HumanMessage(content=user_query)
        ]
    })

    research_text = research_result["messages"][-1].content

    # STEP 2 — Images
    image_result = await image_agent.ainvoke({
        "messages": [
            HumanMessage(
                content=f"""
Generate image suggestions for this topic:

{research_text}
"""
            )
        ]
    })

    image_text = image_result["messages"][-1].content

    # STEP 3 — Writing
    final_result = await writer_agent.ainvoke({
        "messages": [
            HumanMessage(
                content=f"""
Research:

{research_text}

Images:

{image_text}

Create the final structured report.
"""
            )
        ]
    })

    return {
        "research": research_text,
        "images": image_text,
        "final_answer": final_result["messages"][-1].content
    }

In [ ]:
from langchain.agents import create_agent

orchestrator = create_agent(
    model="llama-3.3-70b-versatile",
    tools = [research_tool,image_tool ,writer_tool],
    state_schema=ResearchState,
    system_prompt="""
    You are a prominent historian agent with a deep understanding of Palestinian history and current events in Palestine.
    You possess certain tools that you must utilize to achieve the best, 
    most accurate, and most documented answer you can provide to the user's question.
    You should also give him a picture to reinforce his understanding of what you explained.
    """
)

## Test

In [ ]:
from langchain.messages import HumanMessage

response = await orchestrator.ainvoke(
    {
        "messages": [HumanMessage(content="اريد ان اعلم اكثر عن الوضع الحالي لفلسطين")],
    },
    config={"tags": ["WP"], "recursion_limit": 40},  #tag traces to make them easy to find in Langsmith. Increase number of steps the agent can take to 40.
)

In [ ]:
from pprint import pprint

pprint(response)